# Age- and sex-specific cancer prevalence and crude rates in Our Future Health

## Purpose
This notebook extracts self-reported and inpatient EHR-linked cancer diagnoses from Our Future Health (OFH) and computes age- and sex-specific cancer prevalence counts and crude rates (per 100,000 participants) among OFH participants.

## Outputs
- `outputs/intermediate/questionnaire_diag_fields_metadata.csv`  
  Metadata describing questionnaire diagnosis fields used for cancer case definitions.
- `outputs/raw/questionnaire_diag_fields_raw_values_query.sql`  
  SQL query used to extract raw questionnaire diagnosis responses.
- `results.csv`  
  Aggregated age- and sex-specific cancer counts and crude rates (per 100,000) for selected cancer groupings, structured for direct use in Supplementary Table S11.

## Relationship to manuscript
Outputs from this notebook are used to populate **Supplementary Table S11** (*Supplementary Table S11: Comparison of age- and sex-specific self-reported and EHR cancer prevalence and crude rate (per 100 000) among Our Future Health participants and the population of England (2022)*).

## Data and access notes
Analyses use restricted Our Future Health data accessed within the OFH Trusted Research Environment under approved study permissions. The analysis is restricted to participants aged 20 years or older and to registrations prior to June 2024, corresponding to participants registered in England. All outputs are aggregated, non-disclosive summary statistics, consistent with OFH Safe Output requirements.

## Notes
Cancer groupings are derived from self-reported questionnaire diagnosis fields.


## Setup env

In [ ]:
# Import packages
import dxpy
import shlex
import subprocess
import numpy as np
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

# Import phenofhy
import phenofhy

### Initialize Spark

In [ ]:
spark = SparkSession.builder \
    .appName("Phenotype Analysis") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.kryoserializer.buffer.max", "128") \
    .getOrCreate()

### Table S11

#### Extraction

In [ ]:
phenofhy.load.field_list(
    fields=[
        "participant.birth_month",
        "participant.birth_year",
        "participant.registration_month",
        "participant.registration_year",
        "participant.demog_sex_2_1",
        "participant.demog_sex_1_1",
        "participant.pid",
        "questionnaire.diag_1_m",
        "questionnaire.diag_2_m",
        "questionnaire.diag_cancer_1_m",
        "participant.demog_sex_2_1",
        "participant.demog_sex_1_1",
    ],
    output_file="outputs/intermediate/questionnaire_diag_fields_metadata.csv"
)

phenofhy.extract.fields(
    input_file="outputs/intermediate/questionnaire_diag_fields_metadata.csv",
    output_file="outputs/raw/questionnaire_diag_fields_raw_values_query.sql",
    cohort_key="FULL_SAMPLE_ID",
    sql_only=True
)

raw_diag_df = phenofhy.extract.sql_to_pandas(
    "outputs/raw/questionnaire_diag_fields_raw_values_query.sql"
)

p_df = phenofhy.process.participant_fields(
    raw_diag_df, 
    derive='auto',
    min_age=20,   
    age_group_bins = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, float("inf")],
    age_group_labels = [
        "20–24","25–29","30–34","35–39","40–44",
        "45–49","50–54","55–59","60–64","65–69",
        "70–74","75–79","80–84","85-90", "90+"
    ],
    extra_ranges={"derived.registration_date": (pd.Timestamp.min, pd.Timestamp("2024-06-01"))} # i.e., excl. Scotland clinics)
)

can_df = phenofhy.process.questionnaire_fields(p_df)


# Remove participants who only answered v1, so you can disaggregate cancer
can_df = can_df.loc[can_df['questionnaire.diag_2_m'].notnull()]

#### Analysis

In [ ]:
# compact version: builds item/sex/age rows with counts + rate per 100k (subgroup denominator)
pairs = [("All cancers combined", None), ("Breast", "breast"), ("Colon/rectal", "colon|rectal"),
         ("Prostate", "prostate"), ("Lung or bronchial", "lung|bronchial")]

sex_specs = [([2], "Female"), ([1], "Male"), (list(can_df["derived.sex"].dropna().unique()), "All")]

# age groups ordering
from pandas.api.types import CategoricalDtype

if isinstance(can_df["derived.age_group"].dtype, CategoricalDtype):
    age_groups = list(can_df["derived.age_group"].cat.categories)
else:
    age_groups = sorted(can_df["derived.age_group"].dropna().unique(), key=lambda x: str(x))

rows = []

import numpy as np

valid_non_skin = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20,21,22}
MELANOMA = 1

def has_cancer(row):
    diag = row["questionnaire.diag_cancer_1_m"]
    skin = row["questionnaire.diag_cancer_skin_1_m"]

    # handle missing
    if diag is None or (isinstance(diag, float) and np.isnan(diag)):
        return 0

    # ensure iterable
    diag = np.atleast_1d(diag)
    skin = np.atleast_1d(skin) if skin is not None else np.array([])

    # any non-skin cancer
    if any(code in valid_non_skin for code in diag):
        return 1

    # melanoma only
    if 18 in diag and MELANOMA in skin:
        return 1

    return 0

can_df["sr_cancer"] = can_df.apply(has_cancer, axis=1).astype(int)

def has_code(row, code):
    diag = row["questionnaire.diag_cancer_1_m"]
    if diag is None or (isinstance(diag, float) and np.isnan(diag)):
        return 0
    diag = np.atleast_1d(diag)
    return int(code in diag)

# flags (one per person)
can_df["sr_breast"] = can_df.apply(lambda r: has_code(r, 4), axis=1)
can_df["sr_colorectal"] = can_df.apply(lambda r: has_code(r, 6), axis=1)
can_df["sr_prostate"] = can_df.apply(lambda r: has_code(r, 17), axis=1)
can_df["sr_lung"] = can_df.apply(lambda r: has_code(r, 13), axis=1)


for vals, sex_label in sex_specs:
    for ag in age_groups:
        sub = can_df.loc[
            can_df["derived.sex"].isin(vals) &
            (can_df["derived.age_group"] == ag)
        ]

        pop = len(sub)

        counts = {
            "All cancers combined": int(sub["sr_cancer"].sum()),
            "Breast": int(sub["sr_breast"].sum()),
            "Colon/rectal": int(sub["sr_colorectal"].sum()),
            "Prostate": int(sub["sr_prostate"].sum()),
            "Lung or bronchial": int(sub["sr_lung"].sum()),
        }

        for item_name, _ in pairs:
            cnt = counts[item_name]
            rate = round((cnt / pop * 100_000) if pop > 0 else 0.0, 1)

            rows.append({
                "item": item_name,
                "sex": sex_label,
                "age group": ag,
                "count": cnt,
                "rate": rate
            })

result_df = pd.DataFrame(rows)[["item", "sex", "age group", "count", "rate"]]

# enforce ordering
item_order = ["All cancers combined", "Breast", "Colon/rectal", "Lung or bronchial", "Prostate"]
sex_order = ["Female", "Male", "All"]
age_groups = list(dict.fromkeys(age_groups))

result_df["item"] = pd.Categorical(result_df["item"], categories=item_order, ordered=True)
result_df["sex"] = pd.Categorical(result_df["sex"], categories=sex_order, ordered=True)
result_df["age group"] = pd.Categorical(result_df["age group"], categories=age_groups, ordered=True)

result_df = result_df.sort_values(["item", "sex", "age group"]).reset_index(drop=True)

# Inspect temp results
result_df.to_csv('results.csv')

### EHR analyis

#### Extraction

In [ ]:
phenofhy.load.field_list(
    input_file="inputs/can_nhse_inpat_diag_icd_fields.csv", 
    output_file="outputs/intermediate/can_nhse_inpat_diag_fields_metadata.csv",
)

phenofhy.extract.fields(
    input_file="outputs/intermediate/can_nhse_inpat_diag_fields_metadata.csv",
    output_file="outputs/raw/can_nhse_inpat_diag_fields_raw_values_query.sql", 
    cohort_key="FULL_SAMPLE_ID", 
    sql_only=True
)

raw_nhse_inpat_df = phenofhy.extract.sql_to_pandas(
    "outputs/raw/can_nhse_inpat_diag_fields_raw_values_query.sql"
)

raw_nhse_inpat_df["nhse_eng_inpat.admidate"] = pd.to_datetime(raw_nhse_inpat_df["nhse_eng_inpat.admidate"])

In [ ]:
#### Preprocessing

In [ ]:
# --- Helper functions ---
def icd_range(start, end):
    return [f"C{str(i).zfill(2)}" for i in range(start, end+1)]

EXCLUDE_CODES = {'C44'}                     # Non-melanoma skin cancer
EXCLUDE_PREFIXES = ('C77', 'C78', 'C79')    # Metastases

def is_valid_primary(code):
    if pd.isna(code):
        return False
    code = str(code)
    return (
        code.startswith('C') and
        code[:3] not in EXCLUDE_CODES and
        not code.startswith(EXCLUDE_PREFIXES)
    )

# --- Trait map (cleaned) ---
trait_map = {
    'ehr_cancer':           icd_range(0, 97),   # exclusions handled separately
    'ehr_breast':           ['C50'],
    'ehr_colon_rectal':     ['C18', 'C19', 'C20'],
    'ehr_lung_bronchial':   ['C33', 'C34'],
    'ehr_prostate':         ['C61'],
}

# --- Restrict to PRIMARY DIAGNOSIS ONLY ---
primary_col = 'nhse_eng_inpat.diag_4_01'

# restrict to participants with linked data available
df = raw_nhse_inpat_df.loc[raw_nhse_inpat_df['nhse_eng_inpat.pid'].notnull()].copy()

# flag valid primary cancers (NDRS-like definition)
df['primary_cancer_flag'] = df[primary_col].apply(is_valid_primary)

# keep only valid primary cancer rows
df_primary_cancer = df[df['primary_cancer_flag']].copy()

# --- Restrict to NDRS observation window (maximum prevalence) ---
start_date = pd.Timestamp('1997-03-01')
end_date   = pd.Timestamp('2022-12-31')

df_primary_cancer = df_primary_cancer[
    (df_primary_cancer['nhse_eng_inpat.admidate'] >= start_date) &
    (df_primary_cancer['nhse_eng_inpat.admidate'] <= end_date)
]

# --- FIRST PRIMARY CANCER PER PERSON ---
first_cancer = (
    df_primary_cancer
    .sort_values('nhse_eng_inpat.admidate')
    .drop_duplicates('participant.pid', keep='first')
)

ehr_cancer_pids = set(first_cancer['participant.pid'])

#### Analysis

In [ ]:
# --- STEP 0: build full cohort with age at 2022 index date ---
demo_df = raw_nhse_inpat_df[[
    "participant.pid",
    "participant.birth_year",
    "participant.birth_month",
    "participant.registration_year",
    "participant.registration_month",
    "participant.demog_sex_1_1",
    "participant.demog_sex_2_1"
]].drop_duplicates("participant.pid")

p_full = phenofhy.process.participant_fields(
    demo_df,
    derive='auto',
    min_age=0,
    extra_ranges={"derived.registration_date": (pd.Timestamp.min, pd.Timestamp("2024-06-01"))}
)

# --- Age at NDRS index date ---
INDEX_YEAR = 2022
p_full["age_2022"] = (
    INDEX_YEAR - p_full["participant.birth_year"] -
    (p_full["participant.birth_month"] > 12).astype(int)  # adjust if not yet had birthday by Dec
)

age_bins = [20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,float("inf")]
age_labels = [
    "20–24","25–29","30–34","35–39","40–44",
    "45–49","50–54","55–59","60–64","65–69",
    "70–74","75–79","80–84","85-90","90+"
]

p_full["derived.age_group"] = pd.cut(
    p_full["age_2022"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

# restrict to 20+ and valid sex
p_full = p_full[
    (p_full["age_2022"] >= 20) &
    (p_full["derived.sex"].isin([1, 2]))
].copy()

# --- STEP 1: participant-level dataset ---
p_unique = p_full[[
    "participant.pid",
    "derived.age_group",
    "derived.sex"
]].drop_duplicates().copy()

# --- STEP 2: attach cancer traits ---
# all cancer
p_unique["ehr_cancer"] = p_unique["participant.pid"].isin(ehr_cancer_pids).astype(int)

# site-specific (from cleaned primary cancer dataset)
for trait, codes in trait_map.items():
    if trait == "ehr_cancer":
        continue
    
    valid_pids = set(
        df_primary_cancer[
            df_primary_cancer["nhse_eng_inpat.diag_4_01"].str[:3].isin(codes)
        ]["participant.pid"]
    )
    
    p_unique[trait] = p_unique["participant.pid"].isin(valid_pids).astype(int)

trait_cols = list(trait_map.keys())

# --- STEP 3: counts ---
counts_wide = (
    p_unique
    .groupby(["derived.sex", "derived.age_group"])[trait_cols]
    .sum()
    .reset_index()
)

# --- STEP 4: map sex labels ---
sex_map = {1: "Male", 2: "Female"}
counts_wide["derived.sex"] = counts_wide["derived.sex"].map(sex_map)

# --- STEP 5: add Persons ---
counts_persons = (
    counts_wide
    .groupby("derived.age_group")[trait_cols]
    .sum()
    .reset_index()
)
counts_persons["derived.sex"] = "Persons"

counts_wide = pd.concat([counts_wide, counts_persons], ignore_index=True)

# --- STEP 6: ordering ---
sex_order = ["Male", "Female", "Persons"]
counts_wide["derived.sex"] = pd.Categorical(
    counts_wide["derived.sex"],
    categories=sex_order,
    ordered=True
)

counts_wide = counts_wide.sort_values(["derived.sex", "derived.age_group"])

# --- STEP 7: long format ---
counts_long = counts_wide.melt(
    id_vars=["derived.sex", "derived.age_group"],
    value_vars=trait_cols,
    var_name="trait",
    value_name="n"
)

# --- STEP 8: denominators (aligned population) ---
denoms = (
    p_full
    .groupby(["derived.sex", "derived.age_group"])["participant.pid"]
    .nunique()
    .reset_index(name="denominator")
)

denoms["derived.sex"] = denoms["derived.sex"].map(sex_map)

denoms_persons = (
    denoms
    .groupby("derived.age_group")["denominator"]
    .sum()
    .reset_index()
)
denoms_persons["derived.sex"] = "Persons"

denoms = pd.concat([denoms, denoms_persons], ignore_index=True)

# --- STEP 9: rates ---
rates = counts_long.merge(
    denoms,
    on=["derived.sex", "derived.age_group"],
    how="left"
)

rates["rate_per_100k"] = (rates["n"] / rates["denominator"]) * 100000

### Registry analysis

In [ ]:
#### Extraction

In [ ]:
files = [
    "can_nhse_canreg_pattumour_icd_fields.csv",
]

phenofhy.utils.download_files([
    (str(phenofhy.utils.find_latest_dx_file_id(f)), f"inputs/{f}")
    for f in files
])

pheno_dfs = {f.replace('.csv', ''): pd.read_csv(f'./inputs/{f}') for f in files}

metadata_dfs = phenofhy.load.metadata()

# --- Load and extract registry fields ---
phenofhy.load.field_list(
    input_file="inputs/can_nhse_canreg_pattumour_icd_fields.csv",
    output_file="outputs/intermediate/can_nhse_canreg_pattumour_icd_fields_metadata.csv",
)

phenofhy.extract.fields(
    input_file="outputs/intermediate/can_nhse_canreg_pattumour_icd_fields_metadata.csv",
    output_file="outputs/raw/can_nhse_canreg_pattumour_icd_fields_raw_values_query.sql",
    cohort_key="FULL_SAMPLE_ID",
    sql_only=True,
)

raw_canreg_df = phenofhy.extract.sql_to_pandas(
    "outputs/raw/can_nhse_canreg_pattumour_icd_fields_raw_values_query.sql"
)

In [ ]:
#### Preprocessing

In [ ]:
# filter to questionnaire submissions before cancer linkage cutoff
raw_canreg_df = raw_canreg_df.loc[(raw_canreg_df['participant.registration_year'] <= 2024) & (raw_canreg_df['participant.registration_month'] <= 10)]

canreg_df = raw_canreg_df.loc[raw_canreg_df['participant_nhs_linked.pid'].notnull()]

EXCLUDE_CODES = {'C44'}  # NMSC only; C77–C79 will then be included in "all cancers combined" matching NDRS

def is_valid_primary(code):
    if pd.isna(code):
        return False
    code = str(code)
    return (
        code.startswith('C') and
        code[:3] not in EXCLUDE_CODES
    )

# --- Parse diagnosis date ---
canreg_df["nhse_eng_canreg_pattumour.diagnosisdatebest"] = pd.to_datetime(
    canreg_df["nhse_eng_canreg_pattumour.diagnosisdatebest"]
)

# --- Restrict to NDRS observation window ---
start_date = pd.Timestamp('1995-01-01')
end_date   = pd.Timestamp('2022-12-31')

df_reg = canreg_df[
    (canreg_df["nhse_eng_canreg_pattumour.diagnosisdatebest"] >= start_date) &
    (canreg_df["nhse_eng_canreg_pattumour.diagnosisdatebest"] <= end_date)
].copy()

# --- Apply validity filter (C00-C97 excl. NMSC, excl. metastases) ---
site_col = "nhse_eng_canreg_pattumour.site_icd10_o2"

df_reg["primary_cancer_flag"] = df_reg[site_col].apply(is_valid_primary)
df_reg_primary = df_reg[df_reg["primary_cancer_flag"]].copy()  # filter on flag first
df_reg_primary = df_reg_primary[
    df_reg_primary["nhse_eng_canreg_pattumour.behaviour_icd10_o2"] == "3"
].copy()

# --- First primary cancer per person (all cancers combined) ---
first_cancer_reg = (
    df_reg_primary
    .sort_values("nhse_eng_canreg_pattumour.diagnosisdatebest")
    .drop_duplicates("nhse_eng_canreg_pattumour.pid", keep="first")
)

reg_cancer_pids = set(first_cancer_reg["nhse_eng_canreg_pattumour.pid"])

# --- Site-specific PIDs ---
site_codes = {
    "reg_breast":       ["C50"],
    "reg_colon_rectal": ["C18", "C19", "C20"],
    "reg_lung":         ["C33", "C34"],
    "reg_prostate":     ["C61"],
}

In [ ]:
#### Analysis

In [ ]:
# --- Build denominator from registry extract ---
demo_reg_df = canreg_df[[
    "participant.pid",
    "participant.birth_year",
    "participant.birth_month",
    "participant.registration_year",
    "participant.registration_month",
    "participant.demog_sex_1_1",
    "participant.demog_sex_2_1"
]].drop_duplicates("participant.pid")

p_reg_full = phenofhy.process.participant_fields(
    demo_reg_df,
    derive='auto',
    min_age=0,
    extra_ranges={"derived.registration_date": (pd.Timestamp.min, pd.Timestamp("2024-06-01"))}
)

# --- Age at NDRS index date ---
p_reg_full["age_2022"] = 2022 - p_reg_full["participant.birth_year"]

p_reg_full["derived.age_group"] = pd.cut(
    p_reg_full["age_2022"],
    bins = [20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,float("inf")],
    labels = [
        "20–24","25–29","30–34","35–39","40–44",
        "45–49","50–54","55–59","60–64","65–69",
        "70–74","75–79","80–84","85-90","90+"
    ],
    right=False
)

p_reg_full = p_reg_full[
    (p_reg_full["age_2022"] >= 20) &
    (p_reg_full["derived.sex"].isin([1, 2]))
].copy()

# --- Participant-level flags ---
p_reg = p_reg_full[["participant.pid", "derived.age_group", "derived.sex"]].drop_duplicates().copy()

p_reg["reg_cancer"] = p_reg["participant.pid"].isin(reg_cancer_pids).astype(int)

for trait, codes in site_codes.items():
    valid_pids = set(
        df_reg_primary[
            df_reg_primary[site_col].str[:3].isin(codes)
        ]["nhse_eng_canreg_pattumour.pid"]
    )
    p_reg[trait] = p_reg["participant.pid"].isin(valid_pids).astype(int)

trait_cols_reg = ["reg_cancer"] + list(site_codes.keys())

# --- Counts ---
sex_map = {1: "Male", 2: "Female"}
p_reg["sex_label"] = p_reg["derived.sex"].map(sex_map)

counts_wide_reg = (
    p_reg
    .groupby(["sex_label", "derived.age_group"])[trait_cols_reg]
    .sum()
    .reset_index()
)

counts_persons_reg = (
    counts_wide_reg
    .groupby("derived.age_group")[trait_cols_reg]
    .sum()
    .reset_index()
)
counts_persons_reg["sex_label"] = "Persons"

counts_wide_reg = pd.concat([counts_wide_reg, counts_persons_reg], ignore_index=True)

# --- Denominators ---
denoms_reg = (
    p_reg_full
    .groupby(["derived.sex", "derived.age_group"])["participant.pid"]
    .nunique()
    .reset_index(name="denominator")
)
denoms_reg["sex_label"] = denoms_reg["derived.sex"].map(sex_map)

denoms_persons_reg = (
    denoms_reg
    .groupby("derived.age_group")["denominator"]
    .sum()
    .reset_index()
)
denoms_persons_reg["sex_label"] = "Persons"

denoms_reg = pd.concat([denoms_reg[["sex_label","derived.age_group","denominator"]], 
                         denoms_persons_reg], ignore_index=True)

# --- Rates ---
counts_long_reg = counts_wide_reg.melt(
    id_vars=["sex_label", "derived.age_group"],
    value_vars=trait_cols_reg,
    var_name="trait",
    value_name="n"
)

rates_reg = counts_long_reg.merge(
    denoms_reg,
    on=["sex_label", "derived.age_group"],
    how="left"
)

rates_reg["rate_per_100k"] = (rates_reg["n"] / rates_reg["denominator"]) * 100_000

### Upload results

In [ ]:
# Upload an entire directory of folders
phenofhy.utils.upload_folders([
    ("phenofhy/", "applets/phenofhy"),
])